# OpenMed 434M: adaptive tuning from Run 4

The three completed first-try experiments are preserved and never rerun. Train/Validation only; V4 is excluded. Each run starts from the pinned original pretrained checkpoint. The script retains one overall best model plus concise results for each run. Adaptive mode tries Runs 4–5, then up to three Validation-guided experiments; it stops after two adaptive runs each improve overall micro F1 by less than 0.001.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%pip install -q --only-binary=sentencepiece transformers==4.51.3 tokenizers==0.21.1 sentencepiece==0.2.2 safetensors==0.5.3 "setuptools<82"


In [ ]:
from pathlib import Path
import os, subprocess, sys
PROJECT=Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-PharmaDetect-SuperClinical-434M')
CONFIGS=[PROJECT/'configs'/name for name in ('run04_lr3e-5_ep8.json','run05_lr4e-5_ep8.json')]
ADAPTIVE=True
ENV=dict(os.environ,PYTHONDONTWRITEBYTECODE='1')
subprocess.run([sys.executable,str(PROJECT/'evaluate.py'),'--self-test'],env=ENV,check=True)
subprocess.run([sys.executable,str(PROJECT/'train.py'),'--check-data','--configs',*map(str,CONFIGS)],env=ENV,check=True)


## Execute experiments

Set RUN_TRAINING=True to launch. For a later explicit config, change CONFIGS and set ADAPTIVE=False. Adaptive mode skips completed Run 4–8 results; incomplete runs are not silently restarted.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    command=[sys.executable,'-u',str(PROJECT/'train.py'),'--configs',*map(str,CONFIGS)]+(['--adaptive'] if ADAPTIVE else [])
    with subprocess.Popen(command,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        for line in process.stdout: print(line,end='',flush=True)
        assert process.wait()==0, 'Training stopped; inspect the error above.'
else:
    print('Training disabled until RUN_TRAINING=True.')


In [ ]:
import json
import pandas as pd
from IPython.display import display
results=[json.loads(path.read_text()) for path in (PROJECT/'outputs').glob('*/validation_results.json')]
results=[r for r in results if r.get('completed')]
results.sort(key=lambda r:(int(r['group'][3:5]) if r['group'].startswith('run') and r['group'][3:5].isdigit() else 0,r['configuration']['learning_rate']))
rows=[dict(run=r['group'],lr=r['configuration']['learning_rate'],max_epochs=r['configuration']['epochs'],best_epoch=r['best_epoch'],**r['metrics']['overall']) for r in results]
display(pd.DataFrame(rows))
for r in results:
    print(r['group'],r['configuration'].get('tuning_reason','Completed first-try experiment'))
    display(pd.DataFrame(r['metrics']['per_label']).T)
selection=PROJECT/'outputs/best_model/selection.json'
if selection.exists():
    print('Overall best:',json.loads(selection.read_text()))
    print('Model:',PROJECT/'outputs/best_model')
